# DigitalTwin: Model improvement review
Saved results first; optional reruns below.

The final comparison evaluates reference XGBoost versus the 3-km context candidate.
The matched benchmark compares RF, GB, XGBoost and LightGBM using spatial CV.

Context uses available same-year predictor imagery, excluding the query cell.
Final evaluation includes test-area predictor imagery, not test targets as inputs.
The spatial holdout has been evaluated previously; do not use it for tuning.


In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib.util

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "scripts/21_matched_model_benchmark.py").is_file(), "Project root not found"

print("Project:", ROOT)
print("Kernel Python:", sys.executable)

packages = ["pandas", "numpy", "yaml", "sklearn", "xgboost", "lightgbm", "joblib"]
for package in packages:
    print(package, "OK" if importlib.util.find_spec(package) else "MISSING")

import pandas as pd
from IPython.display import display


## Select saved runs
The defaults select the latest folder containing a summary file.
Set either path explicitly to review a particular historical run.


In [ ]:
def latest_with(base, pattern, filename):
    paths = sorted(p for p in base.glob(pattern) if (p / filename).is_file())
    return paths[-1] if paths else None

FINAL_RUN = latest_with(ROOT / "models/final_evaluation", "*", "test_summary.csv")
BENCHMARK_RUN = latest_with(
    ROOT / "models/experiments", "matched_benchmark_*", "summary.csv"
)

print("Final evaluation:", FINAL_RUN)
print("Matched benchmark:", BENCHMARK_RUN)

def show_csv(folder, filename):
    if folder is None or not (folder / filename).is_file():
        print("Not found:", filename)
        return None
    frame = pd.read_csv(folder / filename)
    print(filename)
    display(frame)
    return frame


## Final spatial holdout: reference vs 3-km context

In [ ]:
final_scores = show_csv(FINAL_RUN, "test_summary.csv")

if final_scores is not None:
    scores = final_scores.set_index("variant")
    if {"reference", "neighbour_3km"}.issubset(scores.index):
        reference = float(scores.loc["reference", "rmse"])
        candidate = float(scores.loc["neighbour_3km", "rmse"])
        print(f"Reference RMSE: {reference:.5f}")
        print(f"3-km RMSE:      {candidate:.5f}")
        print(f"RMSE reduction: {reference - candidate:.5f}")
        if reference > 0:
            print(f"RMSE reduction (%): {100 * (reference - candidate) / reference:.2f}%")
        print("Positive reduction means lower error with context.")


In [ ]:
show_csv(FINAL_RUN, "test_year_metrics.csv")
show_csv(FINAL_RUN, "test_tail_metrics.csv")


## Matched spatial cross-validation benchmark
Compare cell-only and 3-km context within each model family.
These CV scores and the final holdout scores measure different evaluation sets.


In [ ]:
show_csv(BENCHMARK_RUN, "summary.csv")
show_csv(BENCHMARK_RUN, "fold_std.csv")
show_csv(BENCHMARK_RUN, "paired_context_effect.csv")
show_csv(BENCHMARK_RUN, "year_metrics.csv")


## Run metadata and evaluation policies

In [ ]:
import json

for label, folder in [("Final", FINAL_RUN), ("Benchmark", BENCHMARK_RUN)]:
    if folder is not None and (folder / "metadata.json").is_file():
        metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8-sig"))
        print("\n" + label, folder)
        for key in [
            "python", "versions", "imagery_policy", "context_policy",
            "test_history", "smoke", "test_evaluated", "scope"
        ]:
            if key in metadata:
                print(key + ":", metadata[key])


## Optional reruns
Leave these switches False to review saved results only.
A smoke run checks execution; its scores are not research results.
Full benchmark runs 40 fits. Final comparison refits two models and scores the
previously evaluated holdout. Each script creates a new output folder.


In [ ]:
RUN_SMOKE = False
RUN_FULL_BENCHMARK = False
RUN_FINAL_COMPARISON = False

def run_script(name, *args):
    subprocess.run(
        [sys.executable, "-u", str(ROOT / "scripts" / name), *args],
        cwd=str(ROOT),
        check=True
    )


In [ ]:
if RUN_SMOKE:
    run_script("21_matched_model_benchmark.py", "--smoke", "--threads", "2")
else:
    print("Smoke run disabled.")


In [ ]:
if RUN_FULL_BENCHMARK:
    run_script("21_matched_model_benchmark.py", "--threads", "2")
else:
    print("Full benchmark disabled.")


In [ ]:
if RUN_FINAL_COMPARISON:
    run_script("20_final_test_comparison.py")
else:
    print("Final comparison rerun disabled.")


After a rerun, rerun the saved-run selection cell and result cells to refresh tables.